# Random Tree

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')


## Load Data

In [3]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]


### Danh sách lưu kết quả để xuất CSV

In [4]:
results_list = []

def evaluate_model(model, name, X_split, y_split, split_name):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)."""
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred)
    rec = recall_score(y_split, y_pred)
    f1 = f1_score(y_split, y_pred)
    auc = roc_auc_score(y_split, y_prob)

    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    }

    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return res


In [5]:
# K-FOLD CONFIG
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_kfold(model, name, X_data, y_data, cv_split):
    """Đánh giá K-Fold và lưu kết quả trung bình vào results_list."""
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'auc': 'roc_auc'
    }

    cv_results = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv_split,
        scoring=scoring,
        n_jobs=-1
    )

    res = {
        'Algorithm': name,
        'Split': 'KFold',
        'Accuracy': round(cv_results['test_accuracy'].mean(), 4),
        'Precision': round(cv_results['test_precision'].mean(), 4),
        'Recall': round(cv_results['test_recall'].mean(), 4),
        'F1-Score': round(cv_results['test_f1'].mean(), 4),
        'AUC': round(cv_results['test_auc'].mean(), 4)
    }

    results_list.append(res)
    print(
        f"[{name} | KFold] Acc: {res['Accuracy']:.4f} | Prec: {res['Precision']:.4f} | "
        f"Rec: {res['Recall']:.4f} | F1: {res['F1-Score']:.4f} | AUC: {res['AUC']:.4f}"
    )
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [6]:
print("--- V1: Baseline Random Tree ---")
RandomTree_baseline_model = ExtraTreesClassifier(random_state=42, n_estimators=100)

# Đánh giá K-Fold trước
evaluate_kfold(RandomTree_baseline_model, "V1: Random Tree Baseline", X_train, y_train, kfold)

# Train cố định trên train set rồi đánh giá validation/test
RandomTree_baseline_model.fit(X_train, y_train)
evaluate_model(RandomTree_baseline_model, "V1: Random Tree Baseline", X_valid, y_valid, "Validation")
evaluate_model(RandomTree_baseline_model, "V1: Random Tree Baseline", X_test, y_test, "Test")


--- V1: Baseline Random Tree ---
[V1: Random Tree Baseline | KFold] Acc: 0.9959 | Prec: 0.9918 | Rec: 1.0000 | F1: 0.9958 | AUC: 1.0000
[V1: Random Tree Baseline | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V1: Random Tree Baseline | Test] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000


{'Algorithm': 'V1: Random Tree Baseline',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 1.0,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [7]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10, 20], 'min_samples_split': [2, 5, 10], 'max_features': ['sqrt', 'log2', None]}

grid_search = GridSearchCV(
    ExtraTreesClassifier(random_state=42),
    param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
RandomTree_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")

# Đánh giá K-Fold cho mô hình tốt nhất
evaluate_kfold(RandomTree_tuned_model, "V2: Random Tree Tuned (GridSearch)", X_train, y_train, kfold)

evaluate_model(RandomTree_tuned_model, "V2: Random Tree Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(RandomTree_tuned_model, "V2: Random Tree Tuned (GridSearch)", X_test, y_test, "Test")


--- V2: GridSearchCV Tuning ---
Best Params found by CV: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
[V2: Random Tree Tuned (GridSearch) | KFold] Acc: 0.9973 | Prec: 0.9944 | Rec: 1.0000 | F1: 0.9972 | AUC: 1.0000
[V2: Random Tree Tuned (GridSearch) | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V2: Random Tree Tuned (GridSearch) | Test] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000


{'Algorithm': 'V2: Random Tree Tuned (GridSearch)',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 1.0,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

In [8]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp mô hình chính và KNN làm Base Models
base_estimators = [('rt', ExtraTreesClassifier(random_state=42, n_estimators=100)), ('knn', KNeighborsClassifier(n_neighbors=5))]

RandomTree_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=DecisionTreeClassifier(random_state=42),
    cv=kfold
)

# Đánh giá K-Fold cho stacking
evaluate_kfold(RandomTree_stacking_model, "V3: Random Tree Stacking Ensemble", X_train, y_train, kfold)

RandomTree_stacking_model.fit(X_train, y_train)
evaluate_model(RandomTree_stacking_model, "V3: Random Tree Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(RandomTree_stacking_model, "V3: Random Tree Stacking Ensemble", X_test, y_test, "Test")


--- V3: Ensemble Learning (Stacking) ---
[V3: Random Tree Stacking Ensemble | KFold] Acc: 0.9959 | Prec: 0.9972 | Rec: 0.9936 | F1: 0.9954 | AUC: 0.9955
[V3: Random Tree Stacking Ensemble | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V3: Random Tree Stacking Ensemble | Test] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000


{'Algorithm': 'V3: Random Tree Stacking Ensemble',
 'Split': 'Test',
 'Accuracy': 1.0,
 'Precision': 1.0,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### (5) Lưu kết quả

In [9]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/Random_Tree/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU 1 FILE CSV DUY NHẤT
df_results = pd.DataFrame(results_list)

csv_output = os.path.join(save_path, 'RandomForestClassifier_full_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'RandomForestClassifier_baseline.pkl'), 'wb') as f:
    pickle.dump(RandomTree_baseline_model, f)

with open(os.path.join(save_path, 'RandomForestClassifier_tuned.pkl'), 'wb') as f:
    pickle.dump(RandomTree_tuned_model, f)

with open(os.path.join(save_path, 'RandomForestClassifier_stacking.pkl'), 'wb') as f:
    pickle.dump(RandomTree_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu tất cả kết quả vào: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)


----------------------------------------
✅ Đã lưu tất cả kết quả vào: ../experiment/Random_Tree/RandomForestClassifier_full_results.csv
✅ Đã lưu models tại: ../experiment/Random_Tree/
----------------------------------------


,Algorithm,Split,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: Random Tree Baseline,KFold,0.9959,0.9918,1.0000,0.9958,1.0000
1,V1: Random Tree Baseline,Validation,1.0000,1.0000,1.0000,1.0000,1.0000
2,V1: Random Tree Baseline,Test,1.0000,1.0000,1.0000,1.0000,1.0000
3,V2: Random Tree Tuned (GridSearch),KFold,0.9973,0.9944,1.0000,0.9972,1.0000
4,V2: Random Tree Tuned (GridSearch),Validation,1.0000,1.0000,1.0000,1.0000,1.0000
5,V2: Random Tree Tuned (GridSearch),Test,1.0000,1.0000,1.0000,1.0000,1.0000
6,V3: Random Tree Stacking Ensemble,KFold,0.9959,0.9972,0.9936,0.9954,0.9955
7,V3: Random Tree Stacking Ensemble,Validation,1.0000,1.0000,1.0000,1.0000,1.0000
8,V3: Random Tree Stacking Ensemble,Test,1.0000,1.0000,1.0000,1.0000,1.0000


In [10]:
!jupyter nbconvert --to html RandomForestClassifier.ipynb

[NbConvertApp] Converting notebook RandomForestClassifier.ipynb to html
[NbConvertApp] Writing 309297 bytes to RandomForestClassifier.html
